# ClickHouse Trace Logger

This notebook implements observability and trace logging to ClickHouse for monitoring agent performance.

In [0]:
%pip install --upgrade numpy<2.0 clickhouse-connect --quiet
dbutils.library.restartPython()

In [0]:
import time
from typing import Dict, Any
from clickhouse_connect import get_client

def log_agent_run(state: Dict[str, Any], start_time: float) -> None:
    """
    Log agent execution trace to ClickHouse for observability.
    
    Args:
        state: AgentState dictionary containing query, response, SQL, and safety info
        start_time: Unix timestamp when agent execution started
    """
    try:
        # Calculate latency in milliseconds
        latency_ms = (time.time() - start_time) * 1000
        
        # Validate required fields
        if not state.get("user_query") or not state.get("final_answer"):
            print("⚠️  Skipping log: missing required fields (user_query or final_answer)")
            return
        
        # Connect to ClickHouse using Databricks secrets
        client = get_client(
            host=dbutils.secrets.get(scope="clickhouse", key="host"),
            port=int(dbutils.secrets.get(scope="clickhouse", key="port")),
            username=dbutils.secrets.get(scope="clickhouse", key="user"),
            password=dbutils.secrets.get(scope="clickhouse", key="password"),
            secure=True
        )
        
        # Prepare trace data
        data = [[
            state["user_query"],
            state["final_answer"],
            0,  # tokens_used placeholder (to be implemented with LLM token counting)
            state.get("generated_sql", "N/A"),
            1 if state.get("is_safe") else 0,
            latency_ms
        ]]
        
        # Insert trace into ClickHouse
        client.insert(
            'ai_logs',
            data,
            column_names=['prompt', 'response', 'tokens_used', 'generated_sql', 'is_safe', 'latency_ms']
        )
        
        # Confirmation with metrics
        safety_status = "✅ SAFE" if state.get("is_safe") else "❌ UNSAFE"
        print(f"✅ Trace logged to ClickHouse | Latency: {latency_ms:.2f}ms | Safety: {safety_status}")
        
    except Exception as e:
        print(f"❌ Failed to log trace to ClickHouse: {e}")
    finally:
        # Close connection if it exists
        if 'client' in locals():
            client.close()

print("✅ ClickHouse trace logger ready")
print("\n📖 Usage:")
print("   start = time.time()")
print("   # ... run your agent workflow ...")
print("   log_agent_run(final_state, start)")